# Session 10 — Object Detection & YOLO: Hands-On Lab

**Phase 3 · Core CV Tasks · Week 7**

In this notebook we'll put the theory from our slides into practice:

1. **Implement IoU from scratch** — understand the core metric
2. **Implement NMS from scratch** — see how overlapping boxes get filtered
3. **Run YOLOv8 inference** — detect objects using a pre-trained model
4. **Explore detection results** — bounding boxes, classes, and confidence scores
5. **Compare model variants** — nano vs small vs medium
6. **Train on custom data** — see the Ultralytics training API

---

## 1. Setup & Installation

Let's install the required packages and set up our environment.

In [1]:
# Install ultralytics (includes YOLOv8)
# !pip install ultralytics -q

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")

# Set plot style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

PyTorch version: 2.9.1+cu128
CUDA available:  True
GPU:             NVIDIA GeForce GTX 1660 Ti with Max-Q Design


## 2. Intersection over Union (IoU) — From Scratch

IoU measures how much two bounding boxes overlap. It's the **foundation** of
every detection metric (mAP, NMS threshold, etc.).

$$\text{IoU} = \frac{\text{Area of Intersection}}{\text{Area of Union}}$$

We represent each box as `[x1, y1, x2, y2]` — the top-left and bottom-right corners.

In [ ]:
def compute_iou(box_a, box_b):
    """
    Compute IoU between two bounding boxes.
    
    Args:
        box_a: [x1, y1, x2, y2] — top-left and bottom-right corners
        box_b: [x1, y1, x2, y2]
    
    Returns:
        IoU value (float between 0 and 1)
    """
    # Step 1: Find the intersection rectangle
    x1_inter = max(box_a[0], box_b[0])  # left edge
    y1_inter = max(box_a[1], box_b[1])  # top edge
    x2_inter = min(box_a[2], box_b[2])  # right edge
    y2_inter = min(box_a[3], box_b[3])  # bottom edge
    
    # Step 2: Compute intersection area (0 if no overlap)
    inter_width  = max(0, x2_inter - x1_inter)
    inter_height = max(0, y2_inter - y1_inter)
    intersection = inter_width * inter_height
    
    # Step 3: Compute area of each box
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    
    # Step 4: Union = sum of areas - intersection (avoid double-counting)
    union = area_a + area_b - intersection
    
    # Step 5: IoU
    if union == 0:
        return 0.0
    return intersection / union

### Testing our IoU function

Let's test with different overlap scenarios and visualise them.

In [ ]:
def visualise_iou(box_a, box_b, title=""):
    """Draw two boxes and show their IoU."""
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    
    # Draw box A (blue)
    rect_a = patches.Rectangle(
        (box_a[0], box_a[1]), box_a[2]-box_a[0], box_a[3]-box_a[1],
        linewidth=2.5, edgecolor='dodgerblue', facecolor='dodgerblue', alpha=0.3, label='Box A'
    )
    ax.add_patch(rect_a)
    
    # Draw box B (red)
    rect_b = patches.Rectangle(
        (box_b[0], box_b[1]), box_b[2]-box_b[0], box_b[3]-box_b[1],
        linewidth=2.5, edgecolor='crimson', facecolor='crimson', alpha=0.3, label='Box B'
    )
    ax.add_patch(rect_b)
    
    iou = compute_iou(box_a, box_b)
    ax.set_xlim(-1, 12)
    ax.set_ylim(-1, 12)
    ax.set_aspect('equal')
    ax.legend(fontsize=12)
    ax.set_title(f"{title}\nIoU = {iou:.3f}", fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Test Case 1: Partial overlap
visualise_iou([1, 1, 5, 5], [3, 3, 7, 7], "Partial Overlap")

# Test Case 2: High overlap
visualise_iou([1, 1, 6, 6], [2, 2, 7, 7], "High Overlap")

# Test Case 3: No overlap
visualise_iou([1, 1, 3, 3], [5, 5, 8, 8], "No Overlap")

# Test Case 4: Perfect overlap
visualise_iou([2, 2, 6, 6], [2, 2, 6, 6], "Perfect Match")

### Vectorised IoU (for batches)

In practice, we need to compute IoU for hundreds of boxes at once. Here's the
NumPy-vectorised version:

In [ ]:
def compute_iou_batch(boxes_a, boxes_b):
    """
    Compute IoU between every pair of boxes in two arrays.
    
    Args:
        boxes_a: (N, 4) array of boxes
        boxes_b: (M, 4) array of boxes
    
    Returns:
        (N, M) array of IoU values
    """
    # Expand dims for broadcasting: (N,1,4) vs (1,M,4)
    a = boxes_a[:, np.newaxis, :]  # (N, 1, 4)
    b = boxes_b[np.newaxis, :, :]  # (1, M, 4)
    
    # Intersection
    x1 = np.maximum(a[..., 0], b[..., 0])
    y1 = np.maximum(a[..., 1], b[..., 1])
    x2 = np.minimum(a[..., 2], b[..., 2])
    y2 = np.minimum(a[..., 3], b[..., 3])
    
    intersection = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
    
    area_a = (a[..., 2] - a[..., 0]) * (a[..., 3] - a[..., 1])
    area_b = (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])
    
    union = area_a + area_b - intersection
    return np.where(union > 0, intersection / union, 0.0)

# Quick test
predicted = np.array([[1, 1, 5, 5], [3, 3, 7, 7], [6, 6, 10, 10]])
ground_truth = np.array([[1, 1, 5, 5], [4, 4, 8, 8]])

iou_matrix = compute_iou_batch(predicted, ground_truth)
print("IoU Matrix (predicted × ground_truth):")
print(np.round(iou_matrix, 3))

## 3. Non-Maximum Suppression (NMS) — From Scratch

After a detector runs, it produces many overlapping boxes for the same object.
NMS filters out the duplicates, keeping only the best box per object.

**Algorithm:**
1. Sort all boxes by confidence (highest first)
2. Pick the top box → it's a final detection
3. Compute IoU of all remaining boxes with the picked box
4. Remove any box with IoU > threshold (they detect the same object)
5. Repeat until no boxes remain

In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    """
    Non-Maximum Suppression.
    
    Args:
        boxes:         (N, 4) array — [x1, y1, x2, y2]
        scores:        (N,) array — confidence scores
        iou_threshold: float — suppress boxes with IoU above this
    
    Returns:
        List of indices to keep
    """
    # Sort by confidence (descending)
    order = scores.argsort()[::-1]
    
    keep = []
    
    while len(order) > 0:
        # Step 1: Pick the box with the highest score
        best_idx = order[0]
        keep.append(best_idx)
        
        # Step 2: If only one box left, we're done
        if len(order) == 1:
            break
        
        # Step 3: Compute IoU of the best box with all remaining boxes
        remaining = order[1:]
        ious = np.array([
            compute_iou(boxes[best_idx], boxes[idx])
            for idx in remaining
        ])
        
        # Step 4: Keep only boxes with IoU below the threshold
        mask = ious <= iou_threshold
        order = remaining[mask]
    
    return keep

### Visualising NMS in action

Let's create a scenario with multiple overlapping detections and see how NMS cleans them up.

In [ ]:
def draw_detections(ax, boxes, scores, labels=None, color='lime', title=""):
    """Draw bounding boxes on an axis."""
    for i, (box, score) in enumerate(zip(boxes, scores)):
        rect = patches.Rectangle(
            (box[0], box[1]), box[2]-box[0], box[3]-box[1],
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        label = f"{score:.2f}"
        if labels is not None:
            label = f"{labels[i]}: {label}"
        ax.text(box[0], box[1]-2, label, color=color, fontsize=10,
                fontweight='bold', bbox=dict(boxstyle='round,pad=0.2',
                facecolor='black', alpha=0.7))
    ax.set_title(title, fontsize=14, fontweight='bold')

# Simulate: 3 overlapping detections of a "dog" + 2 overlapping detections of a "car"
# (imagine the detector found these)
boxes = np.array([
    # Dog detections (overlapping)
    [50, 100, 250, 350],   # high confidence
    [55, 105, 255, 360],   # slightly shifted
    [60, 95,  245, 345],   # another duplicate
    # Car detections (overlapping)
    [400, 80, 600, 200],   # high confidence
    [410, 85, 610, 210],   # slightly shifted
])
scores = np.array([0.95, 0.87, 0.72, 0.91, 0.65])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Before NMS
axes[0].set_xlim(0, 700)
axes[0].set_ylim(400, 0)  # Flip y-axis (image coords)
draw_detections(axes[0], boxes, scores, color='yellow',
                title=f"BEFORE NMS — {len(boxes)} boxes")
axes[0].set_facecolor('#1a1a2e')

# After NMS
keep = nms(boxes, scores, iou_threshold=0.5)
axes[1].set_xlim(0, 700)
axes[1].set_ylim(400, 0)
draw_detections(axes[1], boxes[keep], scores[keep], color='lime',
                title=f"AFTER NMS — {len(keep)} boxes kept")
axes[1].set_facecolor('#1a1a2e')

print(f"Boxes before NMS: {len(boxes)}")
print(f"Boxes after NMS:  {len(keep)}")
print(f"Kept indices:     {keep}")

plt.tight_layout()
plt.show()

### Effect of IoU threshold

A lower threshold is more aggressive (suppresses more boxes). Let's see:

In [ ]:
thresholds = [0.3, 0.5, 0.7, 0.9]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, thresh in zip(axes, thresholds):
    keep = nms(boxes, scores, iou_threshold=thresh)
    ax.set_xlim(0, 700)
    ax.set_ylim(400, 0)
    ax.set_facecolor('#1a1a2e')
    draw_detections(ax, boxes[keep], scores[keep], color='lime',
                    title=f"IoU thresh = {thresh}\n({len(keep)} boxes)")

plt.suptitle("Effect of NMS IoU Threshold", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. YOLOv8 Inference with Ultralytics

Now let's use a **real** object detector! We'll load a pre-trained YOLOv8 model
and run it on our sample image.

The Ultralytics library makes this incredibly simple — just 3 lines of code.

In [ ]:
from ultralytics import YOLO

# Load a pre-trained YOLOv8 nano model (fastest, smallest)
model = YOLO('yolov8n.pt')

print(f"Model: {model.model_name}")
print(f"Task:  {model.task}")

### Run inference on our sample image

Let's detect objects in the dog + bicycle image from our slides:

In [ ]:
# Run inference on our sample image
results = model('dog_bike_car.jpg')

# The results object contains everything we need
result = results[0]  # First (and only) image

print(f"Image shape: {result.orig_shape}")
print(f"Objects detected: {len(result.boxes)}")
print(f"Inference speed: {result.speed['inference']:.1f}ms")
print()

# Show detected objects
for i, box in enumerate(result.boxes):
    cls_id = int(box.cls[0])
    cls_name = result.names[cls_id]
    confidence = float(box.conf[0])
    coords = box.xyxy[0].tolist()  # [x1, y1, x2, y2]
    
    print(f"  [{i}] {cls_name:12s}  conf={confidence:.3f}  "
          f"  box=[{coords[0]:.0f}, {coords[1]:.0f}, {coords[2]:.0f}, {coords[3]:.0f}]")

### Visualise the detections

Ultralytics provides a built-in `plot()` method that draws boxes, labels, and
confidence scores on the image:

In [ ]:
# Method 1: Ultralytics built-in visualisation
annotated = result.plot()  # Returns a numpy array (BGR)

plt.figure(figsize=(12, 8))
plt.imshow(annotated[..., ::-1])  # Convert BGR → RGB for matplotlib
plt.axis('off')
plt.title("YOLOv8n — Automatic Detection", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### Custom visualisation

Let's also draw the detections ourselves so we understand the data structure:

In [ ]:
# Method 2: Custom drawing for full control
img = Image.open('dog_bike_car.jpg')

fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.imshow(img)

# Define some nice colours
colors = plt.cm.Set2(np.linspace(0, 1, 10))

for i, box in enumerate(result.boxes):
    cls_id = int(box.cls[0])
    cls_name = result.names[cls_id]
    confidence = float(box.conf[0])
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    
    color = colors[cls_id % len(colors)]
    
    # Draw bounding box
    rect = patches.Rectangle(
        (x1, y1), x2-x1, y2-y1,
        linewidth=3, edgecolor=color, facecolor='none'
    )
    ax.add_patch(rect)
    
    # Draw label
    label = f"{cls_name} {confidence:.0%}"
    ax.text(x1, y1-8, label, color='white', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.85))

ax.axis('off')
ax.set_title("YOLOv8n — Custom Visualisation", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Exploring the Results Object

The `Results` object from Ultralytics is rich with information. Let's explore
every attribute you'll commonly use:

In [ ]:
# ── Bounding Boxes ──────────────────────────────────────────
print("=" * 60)
print("BOUNDING BOXES")
print("=" * 60)

# Different box formats
print("\nxyxy format (x1, y1, x2, y2) — top-left and bottom-right:")
print(result.boxes.xyxy)

print("\nxywh format (center_x, center_y, width, height):")
print(result.boxes.xywh)

print("\nNormalised xyxy (0-1 range, relative to image size):")
print(result.boxes.xyxyn)

In [ ]:
# ── Classes & Confidence ────────────────────────────────────
print("=" * 60)
print("CLASSES & CONFIDENCE")
print("=" * 60)

print("\nClass IDs:", result.boxes.cls)
print("Class names:", [result.names[int(c)] for c in result.boxes.cls])
print("Confidence:", result.boxes.conf)

# ── Complete summary ───────────────────────────────────────
print("\n" + "=" * 60)
print("ALL COCO CLASSES (80 total)")
print("=" * 60)
for idx, name in result.names.items():
    print(f"  {idx:3d}: {name}", end="")
    if (idx + 1) % 5 == 0:
        print()
print()

### Confidence filtering

You can control which detections are returned by setting a confidence threshold:

In [ ]:
# Run with different confidence thresholds
thresholds = [0.1, 0.25, 0.5, 0.75]

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, conf_thresh in zip(axes, thresholds):
    res = model('dog_bike_car.jpg', conf=conf_thresh, verbose=False)
    annotated = res[0].plot()
    ax.imshow(annotated[..., ::-1])
    ax.set_title(f"conf={conf_thresh}\n({len(res[0].boxes)} detections)", fontsize=12)
    ax.axis('off')

plt.suptitle("Effect of Confidence Threshold", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Comparing YOLOv8 Model Variants

YOLOv8 comes in 5 sizes: **nano (n)**, **small (s)**, **medium (m)**, **large (l)**,
and **extra-large (x)**. Larger models are more accurate but slower.

Let's compare nano vs small vs medium on our image:

In [ ]:
import time

model_names = ['yolov8n.pt', 'yolov8s.pt', 'yolov8m.pt']
model_labels = ['YOLOv8n (Nano)', 'YOLOv8s (Small)', 'YOLOv8m (Medium)']

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, mname, mlabel in zip(axes, model_names, model_labels):
    m = YOLO(mname)
    
    # Warm-up run
    _ = m('dog_bike_car.jpg', verbose=False)
    
    # Timed run
    start = time.time()
    res = m('dog_bike_car.jpg', verbose=False)
    elapsed = (time.time() - start) * 1000
    
    annotated = res[0].plot()
    ax.imshow(annotated[..., ::-1])
    
    n_det = len(res[0].boxes)
    ax.set_title(f"{mlabel}\n{n_det} detections · {elapsed:.0f}ms", fontsize=12)
    ax.axis('off')

plt.suptitle("YOLOv8 Model Size Comparison", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. YOLOv8 Beyond Detection

YOLOv8 isn't just for object detection — the same architecture supports
**classification**, **segmentation**, **pose estimation**, and **oriented bounding boxes**.

Let's try segmentation:

In [ ]:
# Load a segmentation model (same architecture, different head)
seg_model = YOLO('yolov8n-seg.pt')

# Run segmentation
seg_results = seg_model('dog_bike_car.jpg', verbose=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Detection
det_results = model('dog_bike_car.jpg', verbose=False)
axes[0].imshow(det_results[0].plot()[..., ::-1])
axes[0].set_title("Detection (yolov8n)", fontsize=14)
axes[0].axis('off')

# Segmentation
axes[1].imshow(seg_results[0].plot()[..., ::-1])
axes[1].set_title("Segmentation (yolov8n-seg)", fontsize=14)
axes[1].axis('off')

plt.suptitle("Detection vs Instance Segmentation", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Understanding mAP in Practice

Let's compute **Precision**, **Recall**, and **AP** step-by-step for a toy example
to see how mAP works.

**Scenario:** We have 5 ground-truth "dog" objects and our detector made 8 predictions
at various confidence levels.

In [ ]:
# Simulate detector output for class "dog"
# Each prediction: (confidence, is_correct_detection)
# A prediction is "correct" (TP) if it matches a ground truth with IoU ≥ 0.5
predictions = [
    (0.95, True),    # TP — matches GT dog #1
    (0.91, True),    # TP — matches GT dog #2
    (0.87, False),   # FP — no matching GT (false alarm)
    (0.82, True),    # TP — matches GT dog #3
    (0.76, False),   # FP — duplicate detection of dog #1
    (0.68, True),    # TP — matches GT dog #4
    (0.55, False),   # FP — background false positive
    (0.42, True),    # TP — matches GT dog #5
]
n_ground_truth = 5

# Sort by confidence (already sorted, but let's be explicit)
predictions.sort(key=lambda x: x[0], reverse=True)

# Compute precision and recall at each step
precisions = []
recalls = []
tp_cumulative = 0
fp_cumulative = 0

print(f"{'Step':>4} {'Conf':>6} {'TP/FP':>6} {'Cum TP':>7} {'Cum FP':>7} {'Prec':>7} {'Recall':>7}")
print("-" * 55)

for i, (conf, is_tp) in enumerate(predictions):
    if is_tp:
        tp_cumulative += 1
    else:
        fp_cumulative += 1
    
    precision = tp_cumulative / (tp_cumulative + fp_cumulative)
    recall = tp_cumulative / n_ground_truth
    precisions.append(precision)
    recalls.append(recall)
    
    status = "TP" if is_tp else "FP"
    print(f"{i+1:4d} {conf:6.2f} {status:>6} {tp_cumulative:7d} {fp_cumulative:7d} "
          f"{precision:7.3f} {recall:7.3f}")

In [ ]:
# Plot the Precision-Recall curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw P-R curve
axes[0].plot(recalls, precisions, 'b-o', markersize=8, linewidth=2)
axes[0].fill_between(recalls, precisions, alpha=0.15, color='blue')
axes[0].set_xlabel('Recall', fontsize=13)
axes[0].set_ylabel('Precision', fontsize=13)
axes[0].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
axes[0].set_xlim(0, 1.05)
axes[0].set_ylim(0, 1.05)
axes[0].grid(True, alpha=0.3)

# Compute AP using 11-point interpolation (PASCAL VOC style)
recall_levels = np.linspace(0, 1, 11)
ap_precisions = []

for r_level in recall_levels:
    # Max precision at recall >= r_level
    precs_at_r = [p for p, r in zip(precisions, recalls) if r >= r_level]
    ap_precisions.append(max(precs_at_r) if precs_at_r else 0)

AP = np.mean(ap_precisions)

axes[1].bar(recall_levels, ap_precisions, width=0.08, color='dodgerblue',
            edgecolor='navy', alpha=0.7)
axes[1].set_xlabel('Recall Level', fontsize=13)
axes[1].set_ylabel('Interpolated Precision', fontsize=13)
axes[1].set_title(f'11-Point Interpolation — AP = {AP:.3f}', fontsize=14, fontweight='bold')
axes[1].set_xlim(-0.05, 1.05)
axes[1].set_ylim(0, 1.1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAverage Precision (AP) for class 'dog': {AP:.3f}")
print(f"\nIn practice, mAP = mean of AP across ALL classes.")

## 9. Training YOLOv8 on Custom Data (Preview)

Training your own YOLO model requires a **dataset in YOLO format**. Here's
what the workflow looks like — we'll do a full training lab in the next session.

### Dataset format

YOLO expects images and labels organised like this:

```
dataset/
├── images/
│   ├── train/
│   │   ├── img001.jpg
│   │   └── img002.jpg
│   └── val/
│       ├── img003.jpg
│       └── img004.jpg
├── labels/
│   ├── train/
│   │   ├── img001.txt    ← one .txt per image
│   │   └── img002.txt
│   └── val/
│       ├── img003.txt
│       └── img004.txt
└── dataset.yaml          ← configuration file
```

Each label file has one line per object:  
`class_id  center_x  center_y  width  height` (all normalised 0-1)

In [ ]:
# Example: dataset.yaml content
dataset_yaml = """
# dataset.yaml
path: ./dataset
train: images/train
val: images/val

names:
  0: cat
  1: dog
  2: bird
"""

print("Example dataset.yaml:")
print(dataset_yaml)

print("=" * 50)
print("Example label file (one object per line):")
print("=" * 50)
print("# class_id  center_x  center_y  width  height")
print("1  0.45  0.52  0.35  0.68    ← dog at centre")
print("0  0.78  0.31  0.22  0.40    ← cat on the right")

In [ ]:
# Training API preview (don't run — for reference)
# This is what you'd run in the next session:

print("=" * 50)
print("TRAINING COMMANDS")
print("=" * 50)

print("""
# ── Python API ──────────────────────────────────
from ultralytics import YOLO

model = YOLO('yolov8n.pt')           # Start from pretrained
results = model.train(
    data='dataset.yaml',              # Your dataset config
    epochs=100,                       # Training epochs
    imgsz=640,                        # Image size
    batch=16,                         # Batch size
    name='my_detector',               # Run name
    patience=20,                      # Early stopping
)

# ── CLI ─────────────────────────────────────────
# yolo train model=yolov8n.pt data=dataset.yaml epochs=100 imgsz=640

# ── Validate ────────────────────────────────────
# metrics = model.val()
# print(f"mAP@50:    {metrics.box.map50:.3f}")
# print(f"mAP@50-95: {metrics.box.map:.3f}")

# ── Export ──────────────────────────────────────
# model.export(format='onnx')     # For deployment
# model.export(format='torchscript')
""")

## 10. Summary & Key Takeaways

### What we built today:

| Concept | What we did |
|---------|------------|
| **IoU** | Implemented from scratch — single pair & vectorised batch |
| **NMS** | Implemented from scratch — saw how threshold affects filtering |
| **YOLOv8 Inference** | Loaded pre-trained model, ran detection in 3 lines |
| **Results Exploration** | Extracted boxes, classes, confidence; multiple formats |
| **Model Comparison** | Compared nano/small/medium speed vs accuracy |
| **Segmentation** | Swapped to YOLOv8-seg with one line change |
| **mAP Computation** | Calculated precision, recall, and AP step-by-step |
| **Training Preview** | Dataset format, training API, and CLI commands |

### Next session: **YOLO Training Lab**
- Prepare a custom dataset using **Roboflow**
- Train YOLOv8 on your own data
- Evaluate with mAP, confusion matrix, and per-class analysis

---
*Session 10 · Phase 3 — Core CV Tasks · Week 7*